# Session B — exposure and calendar figures

Run first: `python -m archive.impact_eda.impact_session_b --through 22`

Outputs: `eda/outputs/impact_eda/figures/session_b/`

In [ ]:
from pathlib import Path
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from processing_data.paths import COURSE_RAW

ROOT = Path.cwd()
if not (ROOT / "archive" / "impact_eda" / "impact_session_b.py").exists():
    ROOT = ROOT.parents[2]
OUT = ROOT / "eda" / "outputs" / "impact_eda"
FIG = OUT / "figures" / "session_b"
FIG.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "#f8f8f8"})

def save(fig, name):
    p = FIG / name
    fig.savefig(p, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("saved", p.relative_to(ROOT))

In [ ]:
# Fig 1 — SB-H1: crop share unusual vs recurring (top 20 by unusual crop units, 2023)
exp = pd.read_csv(OUT / "stage19_crop_exposure_county_year.csv")
admin2 = gpd.read_file(COURSE_RAW / "Administrative boundaries" / "ssd_admin2.geojson")[["adm2_pcode", "adm2_name"]]
u23 = exp[(exp.mask_type == "unusual") & (exp.year == 2023)].sort_values("crop_exposed_units", ascending=False).head(20)
codes = u23.adm2_pcode
sub = exp[exp.adm2_pcode.isin(codes) & (exp.year == 2023)]
pivot = sub.pivot(index="adm2_pcode", columns="mask_type", values="crop_share")
pivot = pivot.reset_index().merge(admin2, on="adm2_pcode", how="left")
labels = pivot.adm2_name.fillna(pivot.adm2_pcode)
fig, ax = plt.subplots(figsize=(8, 6))
y = np.arange(len(pivot))
ax.barh(y - 0.2, pivot["unusual"], height=0.35, label="unusual", color="#c0392b")
ax.barh(y + 0.2, pivot["recurring"], height=0.35, label="recurring", color="#2980b9")
ax.set_yticks(y, labels)
ax.set_xlabel("Crop share of (crop + rangeland) exposure units")
ax.set_title("SB-H1: unusual flood pixels are less crop-heavy than recurring (2023 top counties)")
ax.legend()
save(fig, "fig01_crop_share_unusual_vs_recurring.png")

In [ ]:
# Fig 2 — Map crop-exposed units (unusual, latest year), hatch low coverage
exp = pd.read_csv(OUT / "stage19_crop_exposure_county_year.csv")
yr = int(exp[exp.mask_type == "unusual"].year.max())
u = exp[(exp.mask_type == "unusual") & (exp.year == yr)]
gdf = gpd.read_file(COURSE_RAW / "Administrative boundaries" / "ssd_admin2.geojson")
gdf = gdf.merge(u, on="adm2_pcode", how="left")
fig, ax = plt.subplots(figsize=(7, 8))
gdf.plot(column="crop_exposed_units", ax=ax, legend=True, missing_kwds={"color": "#eee"},
         legend_kwds={"label": "crop exposed units", "shrink": 0.6})
low = gdf[gdf.low_tile_coverage.fillna(False)]
low.boundary.plot(ax=ax, color="black", linewidth=0.6, linestyle="--")
ax.set_title(f"Crop exposure at unusual flood pixels ({yr}); dashed = tile coverage < 95%")
ax.axis("off")
save(fig, "fig02_crop_exposure_map.png")

In [ ]:
# Fig 3 — SB-H2 falsification forest (harvest-window coef, pre2022)
fals = pd.read_csv(OUT / "stage20_calendar_falsification.csv")
sub = fals[(fals.spec == "pre2022") & fals.term.str.contains("harv", case=False)]
sub = sub[~sub.term.str.contains("plant", case=False)]
order = ["unusual_fixed", "recurring_placebo", "rain_defined_unusual", "unusual_fixed_plus_detection"]
sub["test_id"] = pd.Categorical(sub.test_id, categories=order, ordered=True)
sub = sub.sort_values("test_id")
fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(sub.test_id, sub.coef, color="#8e44ad")
ax.axvline(0, color="gray")
for i, (_, r) in enumerate(sub.iterrows()):
    ax.text(r.coef, i, f" p={r.pvalue:.3f}", va="center", fontsize=9)
ax.set_xlabel("Within-county coef on log harvest window flood")
ax.set_title("SB-H2: harvest-window signal under falsification tests (pre2022)")
save(fig, "fig03_calendar_falsification_forest.png")

In [ ]:
# Fig 4 — SB-H3 crop vs population exposure (2024 overlap)
prod = pd.read_csv(OUT / "stage21_exposure_product.csv")
snap = prod[prod.year == prod.year.max()].dropna(subset=["crop_exposed_units", "flood_exposed_pop"])
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(np.log1p(snap.crop_exposed_units), np.log1p(snap.flood_exposed_pop), alpha=0.7, c="#27ae60")
rho = pd.read_csv(OUT / "stage21_exposure_redundancy.csv").spearman_crop_vs_pop.iloc[0]
ax.set_xlabel("log1p(crop exposed units)")
ax.set_ylabel("log1p(flood-exposed population)")
ax.set_title(f"SB-H3: crop vs pop exposure (Spearman={rho:.2f})")
admin2 = gpd.read_file(COURSE_RAW / "Administrative boundaries" / "ssd_admin2.geojson")[["adm2_pcode", "adm2_name"]]
snap = snap.merge(admin2, on="adm2_pcode", how="left")
resid = snap.copy()
resid["rank_diff"] = resid.crop_exposed_units.rank() - resid.flood_exposed_pop.rank()
for _, r in resid.reindex(resid.rank_diff.abs().sort_values(ascending=False).index).head(5).iterrows():
    ax.annotate(r.adm2_name or r.adm2_pcode, (np.log1p(r.crop_exposed_units), np.log1p(r.flood_exposed_pop)), fontsize=8)
save(fig, "fig04_crop_vs_pop_scatter.png")

In [ ]:
# Fig 5 — Exposure trajectory top 5 counties (unusual crop units), post-2020 break
exp = pd.read_csv(OUT / "stage19_crop_exposure_county_year.csv")
u = exp[exp.mask_type == "unusual"]
top = u.groupby("adm2_pcode", observed=True).crop_exposed_units.mean().sort_values(ascending=False).head(5).index
admin2 = gpd.read_file(COURSE_RAW / "Administrative boundaries" / "ssd_admin2.geojson")[["adm2_pcode", "adm2_name"]]
fig, ax = plt.subplots(figsize=(9, 4))
for pcode in top:
    s = u[u.adm2_pcode == pcode].sort_values("year")
    name = admin2.loc[admin2.adm2_pcode == pcode, "adm2_name"].iloc[0]
    ax.plot(s.year, s.crop_exposed_units, marker="o", label=name)
ax.axvline(2021.5, color="red", linestyle="--", label="post-2020 regime")
ax.set_ylabel("Crop exposed units (unusual)")
ax.set_title("Top counties: crop flood exposure over time")
ax.legend(fontsize=8, loc="upper left")
save(fig, "fig05_exposure_trajectory_top5.png")